# JavaScript 进阶 - 第15天

本天学习原型对象、对象原型、原型继承、原型链，并使用构造函数和原型方法封装一个消息提示模态框。

## 一、构造函数与原型对象

构造函数中的公共属性适合写在函数内部；公共方法如果也写在构造函数内部，每次 `new` 都会创建一份新函数，浪费内存。每个构造函数都有 `prototype` 属性，指向原型对象。把不变的方法挂在原型对象上，所有实例可以共享同一个函数。

```javascript
function Star(name, age) {
  this.name = name
  this.age = age
}
Star.prototype.sing = function () {
  console.log('唱歌')
}
const ldh = new Star('刘德华', 18)
const zxy = new Star('张学友', 19)
console.log(ldh.sing === zxy.sing) // true
```

原型方法中的 `this` 指向调用它的实例对象。`ldh.sing()` 中的 `this` 是 `ldh`，而不是 `Star.prototype`。

课堂示例：

```html
<!DOCTYPE html>
<html lang="zh-CN">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Document</title>
</head>

<body>
  <script>
    let that
    function Star(name, age) {
      // 1. 公共的属性写到 构造函数 里面
      this.name = name
      this.age = age
      // this.sing = function () {
      //   console.log('唱歌')
      // }
    }

    // 2. 公共的方法写到 原型对象 里面
    // 2.1 构造函数通过原型分配的函数是所有对象所 共享 的
    // 2.2 JavaScript 规定，每一个构造函数都有一个 prototype 属性，指向另一个对象，我们称之为原型对象
    // 2.3 这个对象可以挂载函数，对象实例化不会多次创建原型上函数，节约内存
    // 2.4 我们可以把那些不变的方法，直接定义在 prototype 对象上，这样所有对象的实例就可以共享这些方法
    // 2.5 构造函数和原型对象中的 this 都指向实例化的对象
    Star.prototype.sing = function () {
      that = this
      console.log('唱歌')
    }

    const ldh = new Star('刘德华', 18)
    const zxy = new Star('张学友', 19)
    ldh.sing()
    console.log(ldh === zxy) // false 说明对象不一样
    console.log(ldh.sing === zxy.sing) // true 说明函数一样 共享
    console.log(that === ldh) // true 说明 that 指向了(this, this指向了) ldh
  </script>
</body>

</html>
```

## 二、给数组扩展方法

数组也是对象，可以给 `Array.prototype` 增加公共方法，让所有数组实例都能调用。原型方法必须使用普通函数，这样调用时才能获得正确的 `this`；箭头函数没有自己的 `this`。

```javascript
Array.prototype.sum = function () {
  return this.reduce((pre, cur) => pre + cur, 0)
}
Array.prototype.max = function () {
  return Math.max(...this)
}
```

课堂示例：

```html
<!DOCTYPE html>
<html lang="zh-CN">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Document</title>
</head>

<body>
  <script>
    // 自己定义 数组扩展方法 求和 和 最大值
    // 定义一个方法 任何一个数组实例对象都可以使用
    // 自定义的方法写到 数组.prototype 里面

    // 1. 求和
    // 注意： 对象的方法(包括原型方法)不能使用箭头函数 !!!
    Array.prototype.sum = function () {
      return this.reduce((pre, cur) => pre + cur, 0)
    }

    // 2. 最大值
    Array.prototype.max = function () {
      return Math.max(...this)
    }

    const arr = [1, 2, 3, 4, 5]
    console.log(arr.sum()) // 15
    console.log(arr.max()) // 5
  </script>
</body>

</html>
```

实际开发中要谨慎修改内置原型，避免和第三方库或未来标准方法重名。练习时可以使用独立工具函数代替全局扩展。

## 三、constructor 属性

原型对象默认有 `constructor` 属性，指向创建它的构造函数。如果直接给 `prototype` 重新赋值为一个对象，新的对象默认不再自动指回原构造函数，因此要手动补上：

```javascript
function Star() {}
Star.prototype = {
  constructor: Star,
  sing() { console.log('唱歌') },
  dance() { console.log('跳舞') }
}
```

`constructor` 主要用于确认原型对象所属的构造函数，不负责创建实例。

课堂示例：

```html
<!DOCTYPE html>
<html lang="zh-CN">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Document</title>
</head>

<body>
  <script>
    function Star() {

    }
    Star.prototype = {
      // constructor 是原型对象 prototype 的一个属性
      // constructor的作用：指向创造这个原型对象的 构造函数
      // 重新指向创造这个原型对象的 构造函数
      constructor: Star,
      sing: function () {
        console.log('唱歌')
      },
      dance: function () {
        console.log('跳舞')
      }
    }
    console.log(Star.prototype)
  </script>
</body>

</html>
```

## 四、对象原型 `__proto__`

实例对象的 `__proto__` 指向其构造函数的 `prototype`：

```javascript
function Star() {}
const ldh = new Star()
console.log(ldh.__proto__ === Star.prototype) // true
console.log(ldh.__proto__.constructor === Star) // true
```

`__proto__` 主要用于理解原型关系，开发中推荐使用 `Object.getPrototypeOf(obj)` 读取原型，不要随意修改它。

课堂示例：

```html
<!DOCTYPE html>
<html lang="zh-CN">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Document</title>
</head>

<body>
  <script>
    function Star() { }
    const ldh = new Star()
    // 对象原型 __proto__ 只读 在某些浏览器显示的是 [[Prototype]]
    // 实例对象.__proto__ 指向 该实例对象的构造函数的原型对象
    // 实例对象.__proto__constructor 指向 该实例对象的构造函数
    console.log(ldh.__proto__)

    // 构造函数 实例对象 原型对象 三者关系
    // const ldh = new Star() // 通过 实例化 构造函数 得到 实例对象

    console.log(Star.prototype) // Star.prototype 指向 原型对象
    console.log(ldh.__proto__) // ldh.__proto__ 指向 原型对象
    console.log(Star.prototype === ldh.__proto__) // true 

    console.log(ldh.__proto__.constructor) // 指向 构造函数 Star
    console.log(Star.prototype.constructor) // 指向 构造函数 Star
    console.log(ldh.__proto__.constructor === Star) // true 


  </script>
</body>

</html>
```

## 五、原型继承

JavaScript 中可以让子构造函数的原型对象继承父构造函数的实例：

```javascript
function Person() {
  this.eye = 2
  this.head = 1
}
function Woman() {}
Woman.prototype = new Person()
Woman.prototype.constructor = Woman
Woman.prototype.baby = function () {
  console.log('生孩子')
}
```

重新设置原型后，要把 `constructor` 指回子构造函数。不同子类分别使用 `new Person()`，它们的原型对象不是同一个对象，给 `Woman.prototype` 添加的方法不会出现在 `Man.prototype` 上。

课堂示例：

```html
<!DOCTYPE html>
<html lang="zh-CN">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Document</title>
</head>

<body>
  <script>
    // 公共的部分放到原型上
    // 普通对象
    // const Person = {
    //   eye: 2,
    //   head: 1
    // }

    // Woman.prototype = Person  // 对象 = 另一个对象

    // Woman 和 Man 的原型对象是同一个对象
    // 因此 添加 Woman.prototype.baby 方法时 Man 也会有 baby 方法
    // console.log(Woman.prototype === Man.prototype) // true

    // 构造函数 new 出来的实例对象 结构一样 但是 对象 不一样
    function Person() {
      this.eye = 2
      this.head = 1
    }

    function Woman() {

    }
    // 原型继承
    // 父构造函数（父类） 子构造函数（子类）
    // 子类的原型 = new 父类()
    Woman.prototype = new Person()
    // 指回原来的构造函数
    Woman.prototype.constructor = Woman
    // 给 Woman 添加一个方法 baby
    Woman.prototype.baby = function () {
      console.log('生孩子')
    }

    const red = new Woman()
    console.log(red)

    function Man() {

    }
    Man.prototype = new Person()
    Man.prototype.constructor = Man

    const blue = new Man()
    console.log(blue)

    // 继承构造函数，new 出来的实例对象 结构一样 但是 对象 不一样
    // 因此添加 Woman.prototype.baby 方法时 Man 不会有 baby 方法
    console.log(Woman.prototype === Man.prototype) // false
  </script>
</body>

</html>
```

现代代码更常使用 `class extends` 表达继承，但理解原型继承有助于读懂 JavaScript 对象模型。

## 六、原型链

访问对象的属性或方法时，JavaScript 先查对象自身；找不到就沿着 `__proto__` 向上查找，直到 `Object.prototype`，再向上是 `null`。这条查找路径叫原型链。

```text
实例对象 → 构造函数.prototype → Object.prototype → null
```

`instanceof` 用于判断某构造函数的 `prototype` 是否出现在对象的原型链上：

```javascript
const ldh = new Person()
ldh instanceof Person // true
ldh instanceof Object // true
ldh instanceof Array // false
```

课堂示例：

```html
<!DOCTYPE html>
<html lang="zh-CN">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Document</title>
</head>

<body>
  <script>
    // 所有的对象都是 Object 的实例对象
    // function Object() {

    // } 
    function Person() {

    }
    const ldh = new Person()

    // !!! 只要是对象都有 __proto__ 属性 包括原型对象 !!!
    // 只要是原型对象就会有 constructor 属性

    // console.log(ldh.__proto__ === Person.prototype) // true
    console.log(Person.prototype)
    console.log(Person.prototype.__proto__)
    console.log(Person.prototype.__proto__ === Object.prototype) // true

    // 原型链
    // ldh.__proto__ === Person.prototype
    // Person.prototype.__proto__ === Object.prototype
    // Object.prototype.__proto__ === null

    // 原型链其实是一种查找规则
    // 当访问一个对象的属性或方法时，
    // 会先在对象本身上查找
    // 如果没有找到，就会去它的 __proto__ 指向的对象上查找
    // 如果还没有找到，就会继续去 __proto__ 指向的对象上查找
    // 直到找到 Object.prototype 为止
    // 如果还没有找到，就会返回 null

    // instanceof 运算符
    // 用于判断一个对象是否是某个构造函数的实例(在原型链上)
    // 如果是，就会返回 true
    // 如果不是，就会返回 false
    console.log(ldh instanceof Person) // true
    console.log(ldh instanceof Object) // true
    console.log(ldh instanceof Array) // false
    console.log([1, 2, 3] instanceof Array) // true
    console.log(Array instanceof Object) // true


  </script>
</body>

</html>
```

## 七、综合案例：面向对象封装消息提示

需求：点击“删除”或“登录”按钮时，显示不同标题和内容的模态框；点击关闭按钮移除模态框。

实现步骤：

1. `Modal` 构造函数创建盒子并保存到 `this.modalBox`。
2. 构造函数接收标题和内容，通过模板字符串生成内部结构。
3. 将 `open`、`close` 放到 `Modal.prototype`，供所有实例共享。
4. `open` 先移除旧模态框，再把当前盒子追加到 `body`。
5. 关闭按钮使用箭头函数，继承 `open` 方法中的实例 `this`。

课堂完整代码：

```html
<!DOCTYPE html>
<html lang="zh-CN">

<head>
  <meta charset="UTF-8" />
  <meta http-equiv="X-UA-Compatible" content="IE=edge" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>面向对象封装消息提示</title>
  <style>
    .modal {
      width: 300px;
      min-height: 100px;
      box-shadow: 0 0 10px rgba(0, 0, 0, 0.2);
      border-radius: 4px;
      position: fixed;
      z-index: 999;
      left: 50%;
      top: 50%;
      transform: translate3d(-50%, -50%, 0);
      background-color: #fff;
    }

    .modal .header {
      line-height: 40px;
      padding: 0 10px;
      position: relative;
      font-size: 20px;
    }

    .modal .header i {
      font-style: normal;
      color: #999;
      position: absolute;
      right: 15px;
      top: -2px;
      cursor: pointer;
    }

    .modal .body {
      text-align: center;
      padding: 10px;
    }

    .modal .footer {
      display: flex;
      justify-content: flex-end;
      padding: 10px;
    }

    .modal .footer a {
      padding: 3px 8px;
      background: #ccc;
      text-decoration: none;
      color: #fff;
      border-radius: 2px;
      margin-right: 10px;
      font-size: 14px;
    }

    .modal .footer a.submit {
      background-color: #369;
    }
  </style>
</head>

<body>
  <button id="delete">删除</button>
  <button id="login">登录</button>
  <script>
    // 1. Modal 构造函数封装 - 模态框
    function Modal(title = '', content = '') {
      // 1.1 创建 div 标签
      this.modalBox = document.createElement('div')
      // 1.2 给 div 标签添加类名为 modal
      this.modalBox.className = 'modal'
      // 1.3 把 div 标签添加到 body 中 并修改文字内容
      this.modalBox.innerHTML = `
      <div class="header">${title} <i>x</i></div>
      <div class="body">${content}</div>
      `
      // console.log(this.modalBox)
    }

    // 2. Modal 原型方法封装 - 打开模态框
    // 这个方法不要用箭头函数，因为箭头函数没有 this
    Modal.prototype.open = function () {
      // 5. 解决bug - 每次打开模态框都要创建一个新的盒子
      const box = document.querySelector('.modal')
      // if (box) {
      //   box.remove()
      // }
      box && box.remove() // 逻辑与中断 如果前面为 false 就不执行后面的代码
      // 2.1 把模态框添加到 body 中
      document.body.appendChild(this.modalBox)
      // 4. 给关闭按钮添加点击事件 盒子显示出来就可以添加点击事件了
      // 用箭头函数是为了继承 this 指向
      this.modalBox.querySelector('i').addEventListener('click', () => {
        this.close()
      })
    }
    // 调用 Modal 构造函数
    document.querySelector('#delete').addEventListener('click', function () {
      new Modal('温馨提示', '您没有删除操作权限').open()
    })
    document.querySelector('#login').addEventListener('click', function () {
      new Modal('友情提示', '您还没有注册').open()
    })

    // 3. 给构造函数原型对象添加关闭方法
    Modal.prototype.close = function () {
      this.modalBox.remove()
    }
  </script>
</body>

</html>
```

这里的 `open`、`close` 必须使用普通函数，因为需要通过实例调用获得 `this`。事件监听器内部使用箭头函数，是为了继续使用外层 `open` 中的 `this`。

## 八、易错点

1. 公共方法写入构造函数会重复创建，适合放到原型对象。
2. 原型方法用普通函数，箭头函数没有自己的 `this`。
3. 重写 `prototype` 后要恢复 `constructor`。
4. `__proto__` 是实例到原型的连接，`prototype` 是构造函数的属性。
5. 原型链查找先查实例，再逐级查找原型；同名实例属性会覆盖原型属性。
6. `instanceof` 判断的是原型链关系，不是简单比较构造函数名称。

建议独立完成模态框案例，再增加遮罩层、点击确定回调和按 Esc 关闭功能。

## 九、资料来源

来源：[百度网盘分享](https://pan.baidu.com/s/1MxueOHXGQtmE-ypRq1kELA?pwd=9987)，提取码 `9987`。对应目录为 `1. Pink最新版前端JavaScript/3.JS进阶/资料/JavaScript进阶第三天`，课堂代码参考 `04-code` 原始 HTML 示例。